<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/riesgo/notebooks/c4_l6.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C4-L6 · Psicología anti-tilt | 50 días con registro emocional: el estado predice el PnL (+0,65% vs −1,73%).

In [ ]:
# CELDA COLAB-FIRST: correla primero si estas en Google Colab.
# Descarga el CSV del repo; si falla (sin red ), usa el CSV local.
import pandas as pd
from pathlib import Path

ORG = "Emelecto"  # organizacion fija del repo Emelecto/QuantLab
CSV_NOMBRE = "c4_l6.csv"
CSV_URL = f"https://raw.githubusercontent.com/{ORG}/QuantLab/main/web/content/cursos/riesgo/data/{CSV_NOMBRE}"

try:
    df = pd.read_csv(CSV_URL)
    print("CSV descargado desde:", CSV_URL)
except Exception as e:
    print("Uso CSV local (motivo:", str(e)[:80] + ")")
    csv_path = Path("../data") / CSV_NOMBRE
    if not csv_path.exists():
        csv_path = Path(CSV_NOMBRE)  # fallback si corres desde data/
    df = pd.read_csv(csv_path)
print(df.shape)
print(df.head())

In [ ]:
import numpy as np
g = df.groupby("estado")["pnl_pct"].mean()
print(g.round(2))
calma = g["calmado"]; tilt = g["tilt"]
print("Calmado: %+.2f%% | Tilt: %+.2f%%" % (calma, tilt))
print("Un dia en tilt borra %.1f dias calmados" % abs(tilt / calma))

## El tilt se cuenta, no se siente | Cero impulsivas en calma, 3–6 por día en tilt: el termómetro es la conducta.

In [ ]:
import matplotlib.pyplot as plt
orden = ["calmado", "ansioso", "tilt"]
vals = [df[df["estado"] == e]["pnl_pct"].mean() for e in orden]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(orden, vals, color=["#5eead4", "#f59e0b", "#f87171"])
ax.axhline(0, color="#52525b", linewidth=1)
ax.set_ylabel("PnL promedio por dia (%)")
plt.show()

In [ ]:
imp = df.groupby("estado")["impulsivas"].agg(["min", "max"])
print(imp)
print("Regla anti-tilt: 3 perdidas seguidas = pausa 24 h, sin excepciones.")

In [ ]:
# Chequeos automáticos
assert len(df) == 50, "se esperan 50 dias"
assert abs(df[df["estado"] == "calmado"]["pnl_pct"].mean() - 0.65) < 0.01, "calmado +0,65%"
assert abs(df[df["estado"] == "ansioso"]["pnl_pct"].mean() - (-0.19)) < 0.01, "ansioso -0,19%"
assert abs(df[df["estado"] == "tilt"]["pnl_pct"].mean() - (-1.73)) < 0.01, "tilt -1,73%"
assert (df[df["estado"] == "calmado"]["impulsivas"] == 0).all(), "en calma no hay impulsivas"
assert (df[df["estado"] == "tilt"]["impulsivas"] >= 3).all(), "en tilt hay 3+ impulsivas"
print("OK: el estado es el resultado.")